# AI Cycling Coach — GPU Training (Colab)

**Runtime → Change runtime type → T4 GPU** before running.

## Strategy: generate data locally, train here

Generating 50 K athletes takes ~50 min of single-threaded CPU — a waste of
your Colab GPU session. Instead:

1. **On your laptop** run the fast parallel generator (takes ~8 min on 8 cores):
   ```powershell
   cd "c:\Users\yossi\ai coach"
   $env:PYTHONPATH="c:\Users\yossi\ai coach\backend;c:\Users\yossi\ai coach"
   .\.venv\Scripts\python.exe -m ml.training.generate_synthetic `
       --athletes 50000 --output ./ml/data/synthetic.parquet
   ```
2. **Upload the parquet to Google Drive** (drag & drop in drive.google.com).
   Put it at `My Drive/ai-coach-data/synthetic.parquet`.
3. **Run all cells below** — Cell 3 mounts Drive and copies the file in.
   Training starts immediately with no generation overhead.

Steps:
1. Check GPU
2. Clone repo + install deps
3. Mount Drive → copy data
4. Train (~1–2 h on T4)
5. Save model to Drive + push to GitHub

In [ ]:
# ── 1. Check GPU ──────────────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('No GPU — go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── 2. Clone repo ─────────────────────────────────────────────────────────────
import os

REPO = 'https://github.com/yossibello/ai-coach.git'

if not os.path.exists('/content/ai-coach'):
    !git clone {REPO} /content/ai-coach
else:
    !cd /content/ai-coach && git pull

%cd /content/ai-coach

import sys
sys.path.insert(0, '/content/ai-coach/backend')
os.environ['PYTHONPATH'] = '/content/ai-coach/backend'
os.environ['PYTHONIOENCODING'] = 'utf-8'

!mkdir -p ml/data backend/models

In [ ]:
# ── 3. Install dependencies ───────────────────────────────────────────────────
!pip install pandas pyarrow -q
# torch is pre-installed on Colab with CUDA support
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

In [ ]:
# ── 4. Load data from Google Drive ────────────────────────────────────────────
# Data was generated locally (50K athletes, ~8 min on 8 cores) and uploaded
# to Google Drive to avoid burning GPU session time on CPU-only work.
#
# Expected location in Drive:  My Drive/ai-coach-data/synthetic.parquet
# Change DRIVE_PATH below if you put it somewhere else.

from google.colab import drive
drive.mount('/content/drive')

import shutil, os, pandas as pd

DRIVE_PATH = '/content/drive/MyDrive/ai-coach-data/synthetic.parquet'
DATA_FILE  = 'ml/data/synthetic.parquet'

os.makedirs('ml/data', exist_ok=True)

if os.path.exists(DATA_FILE):
    df = pd.read_parquet(DATA_FILE)
    print(f'✓ Data already in /content: {len(df):,} rows, {df.athlete_id.nunique():,} athletes')
elif os.path.exists(DRIVE_PATH):
    print(f'Copying from Drive ({os.path.getsize(DRIVE_PATH)/1e6:.0f} MB)…')
    shutil.copy(DRIVE_PATH, DATA_FILE)
    df = pd.read_parquet(DATA_FILE)
    print(f'✓ Loaded: {len(df):,} rows, {df.athlete_id.nunique():,} athletes')
else:
    raise FileNotFoundError(
        f'\n\nParquet not found at {DRIVE_PATH}\n'
        'Generate it locally first:\n'
        '  .venv\\Scripts\\python.exe -m ml.training.generate_synthetic '
        '--athletes 50000 --output ml/data/synthetic.parquet\n'
        'Then upload to Google Drive → My Drive/ai-coach-data/'
    )

# Sanity check: confirm risk labels are present (added in Scope B)
assert 'risk_ot_class'   in df.columns, 'Old parquet — regenerate with latest generate_synthetic.py'
assert 'risk_inj_target' in df.columns, 'Old parquet — regenerate with latest generate_synthetic.py'
print(f'✓ Risk labels present | Columns: {len(df.columns)}')

In [ ]:
# ── 5. Train ──────────────────────────────────────────────────────────────────
import torch

# Auto-tune batch size based on available VRAM
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
if vram_gb >= 38:
    BATCH_SIZE = 1024   # A100 80 GB / H100
elif vram_gb >= 20:
    BATCH_SIZE = 512    # A100 40 GB / RTX 3090+
else:
    BATCH_SIZE = 256    # T4 16 GB (safe for seq_len=90, input_dim=57)

EPOCHS     = 100
MODEL_FILE = 'backend/models/cycling_coach.pt'

print(f'GPU VRAM: {vram_gb:.1f} GB → batch size: {BATCH_SIZE}')

!python -m ml.training.train \
    --data      {DATA_FILE} \
    --output    {MODEL_FILE} \
    --epochs    {EPOCHS} \
    --batch-size {BATCH_SIZE} \
    --patience  20 \
    2>&1 | tee training.log

In [ ]:
# ── 6a. Save model to Google Drive (survives session end) ─────────────────────
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
dst = '/content/drive/MyDrive/ai-coach-models/'
os.makedirs(dst, exist_ok=True)
shutil.copy('backend/models/cycling_coach.pt', dst)
print('Saved to Google Drive:', dst + 'cycling_coach.pt')

In [ ]:
# ── 6b. Push model back to GitHub ─────────────────────────────────────────────
# You need a GitHub Personal Access Token (PAT) with repo write access.
# Create one at: https://github.com/settings/tokens  (Classic, repo scope)

from getpass import getpass
token = getpass('GitHub PAT (hidden): ')

!git config user.email 'colab@training'
!git config user.name 'Colab Training'
!git remote set-url origin https://{token}@github.com/yossibello/ai-coach.git
!git add backend/models/cycling_coach.pt
!git commit -m "Trained model: {ATHLETES} athletes, {EPOCHS} epochs (Colab GPU)"
!git push origin main
print('Model pushed to GitHub!')

In [ ]:
# ── 7. Quick sanity check ─────────────────────────────────────────────────────
import torch, sys
sys.path.insert(0, '/content/ai-coach/backend')
from app.ml.model import CyclingTransformer

m = CyclingTransformer()
m.load_state_dict(torch.load('backend/models/cycling_coach.pt', map_location='cpu'))
m.eval()
print('Model loaded OK')
print('Params:', sum(p.numel() for p in m.parameters()), )

# Show last training metrics
!tail -10 training.log